Piloto temporal v2 — lectura de artefactos verificados
Orden: preflight → pruebas sintéticas → congelación → piloto serial → recálculo → este notebook.
Entradas: CSV/JSON guardados en fase3/resultados. Este notebook no entrena.
Python3.12.14; sklearn1.9.1; semilla20261010. Panel231 filas; externos165 por procedimiento.
Alcance: fase3 exclusivamente. 2025 exploratorio; no efectos causales ni significancia del5%.

In [1]:
from pathlib import Path
import json, pandas as pd
root=Path.cwd()
if not (root/'experimentos_objetivo3_v2').exists():
    root=next(p for p in Path.cwd().parents if (p/'experimentos_objetivo3_v2').exists())
out=root/'experimentos_objetivo3_v2/fase3/resultados'
verified=json.loads((out/'verificacion_resultados.json').read_text())
assert verified['aprobado']
print(json.dumps({k:verified[k] for k in ('completo','ajustes','segundos_consumidos','saldo','cobertura','mejora_porcentaje','umbral_futuro_satisfecho')},ensure_ascii=False,indent=2))

{
  "completo": true,
  "ajustes": 1552,
  "segundos_consumidos": 1540.906905419055,
  "saldo": 5659.093094580945,
  "cobertura": {
    "media": 165,
    "ols_ambas_00": 165,
    "ols_ipm_00": 165,
    "ols_tasas_00": 165,
    "procedimiento_ambas": 165,
    "procedimiento_ipm": 165,
    "procedimiento_tasas": 165,
    "procedimiento_v2": 165,
    "referencia_v1": 165
  },
  "mejora_porcentaje": 18.392545281085766,
  "umbral_futuro_satisfecho": true
}


Ganadores por ventana y comparación externa
Las selecciones se guardaron antes de predecir cada año externo; la evaluación principal
corresponde al procedimiento completo con posibles configuraciones distintas por año.

In [2]:
print(pd.read_csv(out/'ganadores.csv').to_string(index=False))
metrics=pd.read_csv(out/'metricas.csv')
print(metrics[metrics.ambito.eq('conjunto')].to_string(index=False))
print(metrics[metrics.ambito.eq('anual') & metrics.procedimiento.isin(['procedimiento_v2','referencia_v1'])].to_string(index=False))

 externo            global_id                      ipm                tasas                ambas
    2021 extra_trees_ambas_04               svr_ipm_05 extra_trees_tasas_04 extra_trees_ambas_04
    2022 extra_trees_tasas_04 gradient_boosting_ipm_00 extra_trees_tasas_00 extra_trees_ambas_00
    2023 extra_trees_tasas_00 gradient_boosting_ipm_01 extra_trees_tasas_00 extra_trees_ambas_00
    2024 extra_trees_tasas_00 gradient_boosting_ipm_00 extra_trees_tasas_00 extra_trees_ambas_00
    2025 extra_trees_tasas_04               svr_ipm_04 extra_trees_tasas_00 extra_trees_ambas_00
      procedimiento   ambito grupo   n       mae      rmse     sesgo        r2
              media conjunto todos 165 14.642483 17.555927 -2.808165 -0.020533
       ols_ambas_00 conjunto todos 165  8.097991 10.138072 -1.811655  0.659678
         ols_ipm_00 conjunto todos 165  9.493907 12.251516  1.121005  0.502997
       ols_tasas_00 conjunto todos 165  7.361572  8.975052 -2.413730  0.733281
procedimiento_ambas con

Deterioros, trazabilidad y límites
Métricas territoriales completas en metricas.csv; todas las internas en metricas_internas.csv.
Predicciones con llaves en predicciones_internas.csv y predicciones_externas.csv.
Artefactos por ajuste, transformaciones, advertencias, tiempos/ledger y diez repeticiones
están en resultados/piloto. Recalculo independiente: fuentes/verificar_resultados.py y
enmienda_reporte.py. No se reutilizan métricas históricas ni se recortan predicciones.

In [3]:
deterioros=pd.read_csv(out/'deterioros.csv')
selected=deterioros[deterioros.procedimiento.eq('procedimiento_v2')]
print(selected[selected.ambito.eq('anual')].to_string(index=False))
print('Territorios con mayor MAE v2 que v1:', int(selected[selected.ambito.eq('territorial')].deteriora_mae.sum()))
failures=json.loads((out/'fallos.json').read_text())
print('Ajustes fallidos:',len(failures['ajustes_fallidos']),'Ajustes con advertencias:',len(failures['advertencias']))
print('Advertencias: casos completos/Guainía, procedencia local, comparabilidad IPM, enlace ecológico y dependencia temporal/territorial.')
print('PDF leídos: ninguno. Notebook personal y fase4 pendientes de autorización expresa; no se entregan aquí.')

   procedimiento ambito grupo  delta_mae  delta_rmse  deteriora_mae
procedimiento_v2  anual  2021  -0.148591   -0.000554          False
procedimiento_v2  anual  2022  -1.675486   -1.731409          False
procedimiento_v2  anual  2023  -1.287644   -1.368083          False
procedimiento_v2  anual  2024  -0.982194   -1.369427          False
procedimiento_v2  anual  2025  -1.404815   -1.294385          False
Territorios con mayor MAE v2 que v1: 8
Ajustes fallidos: 0 Ajustes con advertencias: 0
Advertencias: casos completos/Guainía, procedencia local, comparabilidad IPM, enlace ecológico y dependencia temporal/territorial.
PDF leídos: ninguno. Notebook personal y fase4 pendientes de autorización expresa; no se entregan aquí.
